# B2-023 — Practice p18

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** advanced · **Time budget:** 65 minutes  
**Concepts:** denoising-diffusion-probabilistic-models

## Task

Train the unit's tiny ε-prediction DDPM ($T=50$) on the seeded `mixture2d` data and certify the training-loss decrease and the held-out denoising loss.

**Data.** Load the unit loader with the supplied cell. Define `train_rows()` and `heldout_rows()` with no arguments; they return fresh float32 tensors `generative_data.train_tensor("mixture2d")` of shape `(256,2)` and `generative_data.heldout_tensor("mixture2d")` of shape `(64,2)`, built from the immutable `generative_data.TRAIN_IDS["mixture2d"]` and `generative_data.HELDOUT_IDS["mixture2d"]` in stored order. Held-out rows never enter a training call, a `backward()`, an optimizer step, or a hyperparameter choice. The clean training rows reach the model **only** through the `x0` argument of `q_sample`, exactly as returned.

**Schedule and loss.** Implement `make_schedule` and `q_sample` with the p09 contract: `make_schedule(T, beta_1, beta_T)` returns a dict with keys `"beta"`, `"alpha"`, `"alpha_bar"`, `"beta_tilde"`, each a float64 tensor of shape `(T+1,)`: `beta[0] = 0` and `beta[1:] = torch.linspace(beta_1, beta_T, T, dtype=torch.float64)`; `alpha = 1 - beta`; `alpha_bar = torch.cumprod(alpha, dim=0)` (so `alpha_bar[0] = 1`); `beta_tilde[0] = 0` and `beta_tilde[t] = (1 - alpha_bar[t-1]) / (1 - alpha_bar[t]) * beta[t]` for $t\ge1$ (so `beta_tilde[1] = 0`). `q_sample(x0, t, eps, alpha_bar)` returns `alpha_bar[t].to(x0.dtype).unsqueeze(1).sqrt() * x0 + (1 - alpha_bar[t].to(x0.dtype).unsqueeze(1)).sqrt() * eps` for `x0`, `eps` of shape `(B,d)` and an int64 `t` of shape `(B,)` with values in $1,\dots,T$. Set `SCHEDULE = make_schedule(50, 1e-3, 0.2)`. Implement `ddpm_loss(model, x0, t, eps, alpha_bar)` with the p10 contract: `T = alpha_bar.shape[0] - 1`, `x_t = q_sample(x0, t, eps, alpha_bar)` (the module-level `q_sample`, called by name), `eps_hat = model(x_t, (t.to(x0.dtype) / T).unsqueeze(1))`, and the 0-dimensional loss `((eps_hat - eps) ** 2).mean()`.

**Model.** `class Denoiser(nn.Module)` creates exactly `self.net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))`; `forward(x_t, t_scaled)` returns `self.net(torch.cat([x_t, t_scaled], dim=1))` for `x_t` of shape `(B,2)` and `t_scaled` of shape `(B,1)`. Call `torch.manual_seed(SEED)` immediately before `model = Denoiser()`.

**Training.** `optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`. Write `train_ddpm(model, batch, optimizer)` that creates `noise_generator = torch.Generator().manual_seed(SEED)` once and performs **exactly 1500** updates. Each update first draws `t = torch.randint(1, T + 1, (batch.shape[0],), generator=noise_generator)` and then `eps = torch.randn(batch.shape[0], 2, generator=noise_generator)` (in that order, `T = 50`), then runs `optimizer.zero_grad(set_to_none=True)`, forward, loss, `loss.backward()`, `optimizer.step()` with `ddpm_loss(model, batch, t, eps, SCHEDULE["alpha_bar"])` as forward and loss. Full batch, no shuffling, no scheduler, no gradient clipping, no accumulation, no early stopping. It returns the list of 1500 Python-float losses. Call it as `trace = train_ddpm(model, train_rows(), optimizer)`.

**Held-out denoising loss.** Draw once `eval_generator = torch.Generator().manual_seed(SEED + 1)`, `t_eval = torch.randint(1, 51, (64,), generator=eval_generator)`, then `eps_eval = torch.randn(64, 2, generator=eval_generator)`. Under `torch.no_grad()`, `heldout_before` and `heldout_after` are `ddpm_loss(model, heldout_rows(), t_eval, eps_eval, SCHEDULE["alpha_bar"])` before and after training.

**Certify.**
(A) `len(trace) == 1500`, every loss finite.
(B) **Training decrease:** mean of the last 50 losses `<= 0.60 *` the mean of the first 50.
(C) **Held-out:** `heldout_after <= 0.40` and `heldout_after <= 0.5 * heldout_before`.
(D) **Baseline:** compute `baseline = eps_eval.pow(2).mean()`, the loss of a model that always predicts zero, and assert `heldout_after <= 0.5 * baseline`.
(E) At least one parameter changed; `SCHEDULE["alpha_bar"][50]` is below `0.005`.
(F) **Split certificate at the clean-row seam.** For the duration of the training call only, wrap the module-level `q_sample` (recording its `x0` argument) so that it records every row it receives (restore the original afterwards; held-out evaluation later legitimately uses the same seam). Then assert that no recorded row hashes, with `generative_data.row_sha256`, into the held-out part of `generative_data.ROW_SHA256["mixture2d"]` (the hashes at the positions in `generative_data.HELDOUT_IDS["mixture2d"]`), and that the recorded hashes include every training row's hash.
These thresholds were set from the frozen-seed reference run with margin; they are not a priori guarantees.

**Interpretation.** In three sentences, explain why the training loss levels off well above zero, why the held-out loss uses its own fixed `t_eval` and `eps_eval`, and what (D) adds to (C).

**Required answer-check assertions.** Every item (A)–(F), and the shapes `(256,2)` and `(64,2)` of the two splits.

In [ ]:
import importlib.util
import math
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.